# Project-C: NFHS-5 Child-Marriage Analysis

**Purpose:** Reproduce the state-grouped cross-validation experiment for district-level child-marriage prevalence.

**Target variable:** `Women age 20-24 years married before age 18 years (%)`

This notebook calls the same tested functions as `src/grouped_cv.py`. The split is grouped by State/UT so that all districts from a state are held out together. Imputation and scaling are fitted inside each training fold.

## 1. Environment and imports

Install dependencies first with `pip install -r requirements.txt`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src").exists():
    # Supports opening the notebook from the notebooks/ subfolder.
    PROJECT_DIR = PROJECT_DIR.parent

sys.path.insert(0, str(PROJECT_DIR / "src"))
from grouped_cv import load_and_prepare, make_models, run_grouped_cv

DATA_PATH = PROJECT_DIR / "data" / "datafile.csv"
OUTPUT_DIR = PROJECT_DIR / "results"
print("Project directory:", PROJECT_DIR)
print("Dataset exists:", DATA_PATH.exists())

Project directory: /mnt/data/Project-C-reproducible
Dataset exists: True


## 2. Load and inspect the dataset

The preparation function applies the documented feature-screening rules and converts the target to numeric values.

In [2]:
raw = pd.read_csv(DATA_PATH)
print("Raw dataset shape:", raw.shape)
display(raw.head())

Raw dataset shape: (706, 109)


,District Names,State/UT,Number of Households surveyed,Number of Women age 15-49 years interviewed,Number of Men age 15-54 years interviewed,Female population age 6 years and above who ever attended school (%),Population below age 15 years (%),"Sex ratio of the total population (females per 1,000 males)","Sex ratio at birth for children born in the last five years (females per 1,000 males)",Children under age 5 years whose birth was registered with the civil authority (%),Deaths in the last 3 years registered with the civil authority (%),Population living in households with electricity (%),Population living in households with an improved drinking-water source1 (%),Population living in households that use an improved sanitation facility2 (%),Households using clean fuel for cooking3 (%),Households using iodized salt (%),Households with any usual member covered under a health insurance/financing scheme (%),Children age 5 years who attended pre-primary school during the school year 2019-20 (%),Women (age 15-49) who are literate4 (%),Women (age 15-49) with 10 or more years of schooling (%),Women age 20-24 years married before age 18 years (%),Births in the 5 years preceding the survey that are third or higher order (%),Women age 15-19 years who were already mothers or pregnant at the time of the survey (%),Women age 15-24 years who use hygienic methods of protection during their menstrual period5 (%),Current Use of Family Planning Methods (Currently Married Women Age 15-49 years) - Any method6 (%),Current Use of Family Planning Methods (Currently Married Women Age 15-49 years) - Any modern method6 (%),Current Use of Family Planning Methods (Currently Married Women Age 15-49 years) - Female sterilization (%),Current Use of Family Planning Methods (Currently Married Women Age 15-49 years) - Male sterilization (%),Current Use of Family Planning Methods (Currently Married Women Age 15-49 years) - IUD/PPIUD (%),Current Use of Family Planning Methods (Currently Married Women Age 15-49 years) - Pill (%),Current Use of Family Planning Methods (Currently Married Women Age 15-49 years) - Condom (%),Current Use of Family Planning Methods (Currently Married Women Age 15-49 years) - Injectables (%),Total Unmet need for Family Planning (Currently Married Women Age 15-49 years)7 (%),Unmet need for spacing (Currently Married Women Age 15-49 years)7 (%),Health worker ever talked to female non-users about family planning (%),Current users ever told about side effects of current method of family planning8 (%),Mothers who had an antenatal check-up in the first trimester (for last birth in the 5 years before the survey) (%),Mothers who had at least 4 antenatal care visits (for last birth in the 5 years before the survey) (%),Mothers whose last birth was protected against neonatal tetanus (for last birth in the 5 years before the survey)9 (%),Mothers who consumed iron folic acid for 100 days or more when they were pregnant (for last birth in the 5 years before the survey) (%),...,Children Prevalence of symptoms of acute respiratory infection (ARI) in the 2 weeks preceding the survey (Children under age 5 years) (%),Children with fever or symptoms of ARI in the 2 weeks preceding the survey taken to a health facility or health provider (Children under age 5 years) (%),Children under age 3 years breastfed within one hour of birth15 (%),Children under age 6 months exclusively breastfed16 (%),Children age 6-8 months receiving solid or semi-solid food and breastmilk16 (%),"Breastfeeding children age 6-23 months receiving an adequate diet16, 17 (%)","Non-breastfeeding children age 6-23 months receiving an adequate diet16, 17 (%)","Total children age 6-23 months receiving an adequate diet16, 17 (%)",Children under 5 years who are stunted (height-for-age)18 (%),Children under 5 years who are wasted (weight-for-height)18 (%),Children under 5 years who are severely wasted (weight-for-height)19 (%),Children under 5 years who are underweight (weight-for-age)18 (%),C

In [3]:
X, y, groups, feature_cols, audit = load_and_prepare(DATA_PATH)
print(f"Rows: {len(X)} | Predictors: {len(feature_cols)} | States/UTs: {pd.Series(groups).nunique()}")
print("Target summary:")
display(pd.Series(y, name="Child marriage prevalence (%)").describe().to_frame())
print("Audit summary:")
display(pd.Series({k: v for k, v in audit.items() if not isinstance(v, list)}).to_frame("value"))

Rows: 706 | Predictors: 97 | States/UTs: 36
Target summary:


,Child marriage prevalence (%)
count,706.000000
mean,20.446601
std,12.448299
min,0.000000
25%,10.800000
50%,18.250000
75%,28.100000
max,57.600000


Audit summary:


,value
rows,706
states,36
features_used,97
target,Women age 20-24 years married before age 18 ye...
group_column,State
missing_predictor_cells_percent_before_imputation,1.375544
districts_per_state_min,1
districts_per_state_max,75
state_name_normalisation,{'Maharastra': 'Maharashtra'}


## 3. Define models and run GroupKFold

Five folds are used when at least five states are available. The script checks that train/test state sets do not overlap. The baseline predicts the mean target from the training fold.

In [4]:
models = make_models(feature_cols)
results = run_grouped_cv(X, y, groups, models, OUTPUT_DIR)
display(results.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}"}))

,model,n_splits,evaluation,MAE,RMSE,R2
0,Ridge,5,pooled out-of-fold predictions; each state held out once,5.539,6.995,0.684
1,SVR_RBF,5,pooled out-of-fold predictions; each state held out once,6.106,7.795,0.607
2,Random_Forest,5,pooled out-of-fold predictions; each state held out once,6.765,8.502,0.533
3,Training_fold_mean_baseline,5,pooled out-of-fold predictions; each state held out once,10.233,12.497,-0.009


## 4. Inspect fold-level results and out-of-fold predictions

Every district receives one prediction from a model that was trained without any district from that district's state.

In [5]:
fold_metrics = pd.read_csv(OUTPUT_DIR / "grouped_cv_fold_metrics.csv")
oof_predictions = pd.read_csv(OUTPUT_DIR / "grouped_cv_predictions.csv")
display(fold_metrics)
display(oof_predictions.head(10))

,model,fold,train_rows,test_rows,train_states,test_states,state_overlap,MAE,RMSE,R2
0,Ridge,1,564,142,29,7,0,5.699164,6.822308,0.579983
1,Ridge,2,565,141,28,8,0,4.668074,6.118658,0.678814
2,Ridge,3,565,141,29,7,0,6.317686,7.654747,0.766197
3,Ridge,4,565,141,29,7,0,5.930880,7.798413,0.644112
4,Ridge,5,565,141,29,7,0,5.077851,6.422105,0.633136
5,SVR_RBF,1,564,142,29,7,0,6.523752,7.844413,0.444703
6,SVR_RBF,2,565,141,28,8,0,5.301178,6.789720,0.604498
7,SVR_RBF,3,565,141,29,7,0,6.584468,8.184586,0.732711
8,SVR_RBF,4,565,141,29,7,0,6.805675,9.043847,0.521362
9,SVR_RBF,5,565,141,29,7,0,5.311339,6.881841,0.578731


,row_index,state,actual,predicted,model
0,0,Andaman & Nicobar Islands,11.4,9.332249,Ridge
1,1,Andaman & Nicobar Islands,15.4,4.018645,Ridge
2,2,Andaman & Nicobar Islands,17.1,7.279908,Ridge
3,3,Andhra Pradesh,25.4,21.072930,Ridge
4,4,Andhra Pradesh,33.7,33.162909,Ridge
5,5,Andhra Pradesh,25.4,24.079886,Ridge
6,6,Andhra Pradesh,26.0,23.325385,Ridge
7,7,Andhra Pradesh,22.1,30.627811,Ridge
8,8,Andhra Pradesh,25.3,26.096513,Ridge
9,9,Andhra Pradesh,35.4,44.501400,Ridge


## 5. Save the audit record

The audit file records the target, group column, number of states, feature screening and evaluation setup.

In [6]:
import json
audit["n_splits"] = min(5, pd.Series(groups).nunique())
audit["models_evaluated"] = list(models.keys())
audit["baseline"] = "training-fold target mean"
with open(OUTPUT_DIR / "grouped_cv_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)
print("Saved reproducible results to:", OUTPUT_DIR.resolve())

Saved reproducible results to: /mnt/data/Project-C-reproducible/results
